<a href="https://colab.research.google.com/github/Nurdaylight/A-Karpathy-repl/blob/main/RoPE_Tiny_shakespeare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
from labml import experiment
from labml.configs import option, calculate
from labml_nn.transformers import TransformerConfigs
from labml_nn.transformers.basic.autoregressive_experiment import AutoregressiveTransformer, Configs
from labml_helpers.module import Module

In [36]:
from labml_nn.transformers.basic.autoregressive_experiment import Configs
conf = Configs()

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3473: Warning: Overriding option for model: rotary_pe_transformer
  if (await self.run_code(code, result,  async_=asy)):


In [34]:
def _rotary_pe_mha(c: TransformerConfigs):
    from labml_nn.transformers.rope import RotaryPEMultiHeadAttention
    return RotaryPEMultiHeadAttention(c.n_heads, c.d_model, 1.)
calculate(TransformerConfigs.encoder_attn, 'rotary', _rotary_pe_mha)
calculate(TransformerConfigs.decoder_attn, 'rotary', _rotary_pe_mha)
calculate(TransformerConfigs.decoder_mem_attn, 'rotary', _rotary_pe_mha)

@option(Configs.model, 'rotary_pe_transformer')
def _model(c: Configs):
   m = AutoregressiveTransformer(c.transformer.encoder,
                                  c.transformer.src_embed,
                                 c.transformer.generator).to(c.device)

   return m
def main():
    experiment.create(name="rotary_pe_transformer", writers={'screen'})

    experiment.configs(conf, {
        'transformer.src_embed': 'no_pos',
        'transformer.tgt_embed': 'no_pos',
        'transformer.encoder_attn': 'rotary',
        'model': 'rotary_pe_transformer',
        'tokenizer': 'character',
        'prompt_separator': '',
        'prompt': 'It is ',
        'text': 'tiny_shakespeare',
        'seq_len': 512,
        'epochs': 32,
        'batch_size': 4,
        'inner_iterations': 10,
        'd_model': 128,
        'transformer.ffn.d_ff': 512,
        'transformer.n_heads': 16,
        'transformer.dropout': 0.0,
        'optimizer.optimizer': 'Noam',
        'optimizer.learning_rate': 1.,
        'dataloader_shuffle_with_replacement': True
    })

    with experiment.start():
        conf.run()

    # Save final model explicitly (since experiment.add_* is deprecated/absent)
    import torch
    torch.save(conf.model.state_dict(), "rotary_pe_transformer.pt")

In [37]:
if __name__ == "__main__":
    main()

AttributeError: module 'labml.tracker' has no attribute 'set_text'